In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from hyperopt import hp, fmin, tpe
from sklearn.metrics import mean_squared_error

In [2]:
train = pd.read_csv('./elo-merchant-category-recommendation/train.csv')
test = pd.read_csv('./elo-merchant-category-recommendation/test.csv')

In [3]:
train.shape

(201917, 1742)

In [4]:
test.shape

(123623, 1741)

In [13]:
def feature_select_wrapper(train, test):
    print('feature_select_wrapper...')
    label = 'target'
    features = train.columns.tolist()
    features.remove('card_id')
    features.remove('target')

    params_initial = {
        'num_leaves': 31,
        'learning_rate': 0.1,
        'boosting': 'gbdt',
        'min_child_samples': 20,
        'bagging_seed': 2020,
        'bagging_fraction': 0.7,
        'bagging_freq': 1,
        'feature_fraction': 0.7,
        'max_depth': -1,
        'metric': 'rmse',
        'reg_alpha': 0,
        'reg_lambda': 1,
        'objective': 'regression'
    }

    ESR = 30 # early stopping
    NBR = 10000 # iteration times 
    VBE = 50 

    kf = KFold(n_splits = 5, random_state = 2020, shuffle = True)
    fse = pd.Series(0, index=features)

    for train_part_index, eval_index in kf.split(train[features], train[label]):
        train_part = lgb.Dataset(train[features].loc[train_part_index], 
                                 train[label].loc[train_part_index])
        eval = lgb.Dataset(train[features].loc[eval_index], 
                          train[label].loc[eval_index])
        bst = lgb.train(params = params_initial,
                       train_set = train_part,
                       valid_sets = [train_part, eval],
                       valid_names = ['train', 'valid'],
                       num_boost_round = NBR,
                       callbacks=[
                                lgb.early_stopping(stopping_rounds=ESR),
                                lgb.log_evaluation(VBE)
                        ]
        )  
        fse += pd.Series(bst.feature_importance(), index=features)

    feature_select = ['card_id'] + fse.sort_values(ascending=False).index.tolist()[:300]
    print("feature_select_wrapper done!")
    return train[feature_select + ['target']], test[feature_select]

        

In [14]:
train_LGBM, test_LGBM = feature_select_wrapper(train, test)

feature_select_wrapper...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.430133 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 227024
[LightGBM] [Info] Number of data points in the train set: 161533, number of used features: 1626
[LightGBM] [Info] Start training from score -0.390986
Training until validation scores don't improve for 30 rounds
[50]	train's rmse: 3.44373	valid's rmse: 3.70354
Early stopping, best iteration is:
[53]	train's rmse: 3.43619	valid's rmse: 3.70228
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.428725 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 227128
[LightGBM] [Info] Number of data points in the train set: 161533, number of used features: 1629
[LightGBM] [Info] Start training from score -0.396781
Training until validation scores don't improve for 30 rounds
[50]	train's rmse: 3.45

In [15]:
train_LGBM.shape

(201917, 302)

In [28]:
def params_append(params):
    params['feature_pre_filter'] = False
    params['objective'] = 'regression'
    params['metric'] = 'rmse'
    params['bagging_seed'] = 2020
    return params

In [34]:
def param_hyperopt(train):
    label = 'target'
    features = train.columns.tolist()
    features.remove('card_id')
    features.remove('target')

    train_data = lgb.Dataset(train[features], train[label])

    def hyperopt_objective(params):
        params = params_append(params)
        print(params)

        res = lgb.cv(params, train_data, 1000,
                     nfold=2,
                     stratified=False,
                     shuffle=True,
                     callbacks=[lgb.early_stopping(stopping_rounds=20)],
                     seed=2020)
        return min(res['valid rmse-mean'])
    
    params_space = {
        'learning_rate': hp.uniform('learning_rate', 1e-2, 5e-1),
        'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
        'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
        'num_leaves': hp.choice('num_leaves', list(range(10, 300, 10))),
        'reg_alpha': hp.randint('reg_alpha', 0, 10),
        'reg_lambda': hp.uniform('reg_lambda', 0, 10),
        'bagging_freq': hp.randint('bagging_freq', 1, 10),
        'min_child_samples': hp.choice('min_child_samples', list(range(1, 30, 5)))
    }

    params_best = fmin(hyperopt_objective, 
                       params_space, 
                       algo=tpe.suggest, 
                       max_evals=30,
                       rstate=np.random.default_rng(2020))
    
    return params_best

In [35]:
best_clf = param_hyperopt(train_LGBM)

{'bagging_fraction': 0.9429104308567877, 'bagging_freq': 2, 'feature_fraction': 0.5715782198140802, 'learning_rate': 0.21315219327595428, 'min_child_samples': 11, 'num_leaves': 160, 'reg_alpha': 3, 'reg_lambda': 7.561160634893758, 'feature_pre_filter': False, 'objective': 'regression', 'metric': 'rmse', 'bagging_seed': 2020}
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021724 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 66678                    
[LightGBM] [Info] Number of data points in the train set: 100958, number of used features: 300
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084661 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 66678                    
[LightGBM] [Info] Number of data points in the train set: 100958, number of used

In [36]:
best_clf 

{'bagging_fraction': 0.6869739303093313,
 'bagging_freq': 8,
 'feature_fraction': 0.5085673004699267,
 'learning_rate': 0.010396175820347714,
 'min_child_samples': 2,
 'num_leaves': 10,
 'reg_alpha': 7,
 'reg_lambda': 3.1988609813320315}

In [37]:
best_clf = params_append(best_clf)

label = 'target'
features = train_LGBM.columns.tolist()
features.remove('card_id')
features.remove('target')

lgb_train = lgb.Dataset(train_LGBM[features], train_LGBM[label])

In [38]:
bst = lgb.train(best_clf, lgb_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040078 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 66678
[LightGBM] [Info] Number of data points in the train set: 201917, number of used features: 300
[LightGBM] [Info] Start training from score -0.393636


In [39]:
bst.predict(train_LGBM[features])

array([-0.28821605, -1.36811558,  0.02788153, ..., -0.25558873,
       -1.00461926, -0.28821605])

In [40]:
np.sqrt(mean_squared_error(train_LGBM[label], bst.predict(train_LGBM[features])))

3.7293874083738037

In [41]:
test_LGBM['target'] = bst.predict(test_LGBM[features])
test_LGBM[['card_id', 'target']].to_csv('./elo-merchant-category-recommendation/lgbm/submission_LGBM.csv', index=False)

/var/folders/74/684yj_g56d1f10k2kmf73bg00000gn/T/ipykernel_42143/4191009436.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_LGBM['target'] = bst.predict(test_LGBM[features])


In [42]:
test_LGBM[['card_id', 'target']].head()

,card_id,target
0,C_ID_0ab67a22ab,-1.974638
1,C_ID_130fd0cbdd,-0.682038
2,C_ID_b709037bc5,-0.129609
3,C_ID_d27d835a9f,-0.282527
4,C_ID_2b5e3df5c2,-0.401622


In [43]:
def train_predict(train, test, params):
    label = 'target'
    features = train.columns.tolist()
    features.remove('card_id')
    features.remove('target')
    
    params = params_append(params)
    ESR = 30
    NBR = 10000
    VBE = 50

    prediction_test = 0
    cv_score = []
    prediction_train = pd.Series()

    kf = KFold(n_splits=5, random_state=2020, shuffle=True)
    for train_part_index, eval_index in kf.split(train[features], train[label]):
        train_part = lgb.Dataset(train[features].loc[train_part_index], 
                                 train[label].loc[train_part_index])
        eval = lgb.Dataset(train[features].loc[eval_index], 
                          train[label].loc[eval_index])
        bst = lgb.train(params, train_part, valid_sets=[train_part, eval],
                        valid_names=['train', 'valid'],
                        num_boost_round=NBR,
                        callbacks=[lgb.early_stopping(stopping_rounds=ESR),
                                   lgb.log_evaluation(VBE)])
        prediction_test += bst.predict(test[features])
        prediction_train = pd.concat([prediction_train, pd.Series(bst.predict(train[features].loc[eval_index]), index=eval_index)])

        eval_pre = bst.predict(train[features].loc[eval_index])
        score = np.sqrt(mean_squared_error(train[label].loc[eval_index], eval_pre))
        cv_score.append(score)

    print(cv_score, sum(cv_score) / 5)

    pd.Series(prediction_train.sort_index().values).to_csv("./elo-merchant-category-recommendation/lgbm/train_lightgbm.csv", index=False)
    # Save test set predictions to local file
    pd.Series(prediction_test / 5).to_csv("./elo-merchant-category-recommendation/lgbm/test_lightgbm.csv", index=False)
    # Use average score on test set as final model prediction
    test['target'] = prediction_test / 5
    # Save test set predictions in competition format to local file
    test[['card_id', 'target']].to_csv("./elo-merchant-category-recommendation/lgbm/submission_lightgbm.csv", index=False)
    return

In [44]:
train_LGBM, test_LGBM = feature_select_wrapper(train, test)
best_clf = param_hyperopt(train_LGBM)
train_predict(train_LGBM, test_LGBM, best_clf)

feature_select_wrapper...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.423777 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 227024
[LightGBM] [Info] Number of data points in the train set: 161533, number of used features: 1626
[LightGBM] [Info] Start training from score -0.390986
Training until validation scores don't improve for 30 rounds
[50]	train's rmse: 3.44373	valid's rmse: 3.70354
Early stopping, best iteration is:
[53]	train's rmse: 3.43619	valid's rmse: 3.70228
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.358418 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 227128
[LightGBM] [Info] Number of data points in the train set: 161533, number of used features: 1629
[LightGBM] [Info] Start training from score -0.396781
Training until validation scores don't improve for 30 rounds
[50]	train's rmse: 3.45

/var/folders/74/684yj_g56d1f10k2kmf73bg00000gn/T/ipykernel_42143/626201436.py:28: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  prediction_train = pd.concat([prediction_train, pd.Series(bst.predict(train[features].loc[eval_index]), index=eval_index)])


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.129616 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 66412
[LightGBM] [Info] Number of data points in the train set: 161533, number of used features: 300
[LightGBM] [Info] Start training from score -0.396781
Training until validation scores don't improve for 30 rounds
[50]	train's rmse: 3.77698	valid's rmse: 3.75087
[100]	train's rmse: 3.73478	valid's rmse: 3.71294
[150]	train's rmse: 3.70768	valid's rmse: 3.69075
[200]	train's rmse: 3.68777	valid's rmse: 3.67793
[250]	train's rmse: 3.67203	valid's rmse: 3.67017
[300]	train's rmse: 3.65936	valid's rmse: 3.66379
[350]	train's rmse: 3.64918	valid's rmse: 3.65945
[400]	train's rmse: 3.63982	valid's rmse: 3.656
[450]	train's rmse: 3.63161	valid's rmse: 3.65371
[500]	train's rmse: 3.62335	valid's rmse: 3.6511
[550]	train's rmse: 3.6158	valid's rmse: 3.65008
[600]	train's rmse: 3.60852	valid's rmse: 3.6488

/var/folders/74/684yj_g56d1f10k2kmf73bg00000gn/T/ipykernel_42143/626201436.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['target'] = prediction_test / 5
